## loading the models and test them

In [1]:
import json
import time

import joblib
import pandas as pd
import sklearn
import lightgbm

model_folder = "../models/"

print("Looking for the models in:", model_folder)

Looking for the models in: ../models/


In [2]:
with open(model_folder + "model_info.json") as file:
    info = json.load(file)

start = time.time()
model_wifi = joblib.load(model_folder + info["with_wifi"]["file"])
model_no_wifi = joblib.load(model_folder + info["without_wifi"]["file"])
seconds = time.time() - start

wifi_cols = info["with_wifi"]["columns"]
no_wifi_cols = info["without_wifi"]["columns"]

print("Loaded both models in", round(seconds, 2), "seconds")
print("Columns, with wifi   :", len(wifi_cols))
print("Columns, without wifi:", len(no_wifi_cols))

Loaded both models in 0.1 seconds
Columns, with wifi   : 18
Columns, without wifi: 17


#### checking the library versions

In [3]:
print("Versions when the models were saved:", info["versions"])
print()
print("scikit-learn now:", sklearn.__version__, "| same:", sklearn.__version__ == info["versions"]["scikit-learn"])
print("lightgbm now    :", lightgbm.__version__, "| same:", lightgbm.__version__ == info["versions"]["lightgbm"])

Versions when the models were saved: {'python': '3.12.10', 'scikit-learn': '1.9.1', 'lightgbm': '4.7.0', 'pandas': '3.0.6'}

scikit-learn now: 1.9.1 | same: True
lightgbm now    : 4.7.0 | same: True


## Predicting the passengers

In [4]:
passengers = pd.DataFrame(
    [
        # a happy business traveller
        {"customer_type": "Returning", "type_of_travel": "Business", "travel_class": "Business",
         "age": 40, "flight_distance": 1500, "departure_delay": 10,
         "ease_of_online_booking": 4, "checkin_service": 4, "online_boarding": 4,
         "onboard_service": 4, "seat_comfort": 4, "leg_room_service": 4,
         "cleanliness": 4, "food_and_drink": 4, "inflight_service": 4,
         "inflight_wifi_service": 3, "inflight_entertainment": 4, "baggage_handling": 4},
        # an unhappy personal traveller
        {"customer_type": "First-time", "type_of_travel": "Personal", "travel_class": "Economy",
         "age": 22, "flight_distance": 300, "departure_delay": 90,
         "ease_of_online_booking": 1, "checkin_service": 1, "online_boarding": 1,
         "onboard_service": 1, "seat_comfort": 1, "leg_room_service": 1,
         "cleanliness": 1, "food_and_drink": 1, "inflight_service": 1,
         "inflight_wifi_service": 1, "inflight_entertainment": 1, "baggage_handling": 1},
        # someone in the middle: every rating is 3
        {"customer_type": "Returning", "type_of_travel": "Business", "travel_class": "Economy",
         "age": 35, "flight_distance": 800, "departure_delay": 0,
         "ease_of_online_booking": 3, "checkin_service": 3, "online_boarding": 3,
         "onboard_service": 3, "seat_comfort": 3, "leg_room_service": 3,
         "cleanliness": 3, "food_and_drink": 3, "inflight_service": 3,
         "inflight_wifi_service": 3, "inflight_entertainment": 3, "baggage_handling": 3},
    ],
    index=["happy", "unhappy", "middle"],
)

chance_wifi = model_wifi.predict_proba(passengers[wifi_cols])[:, 1]
chance_no_wifi = model_no_wifi.predict_proba(passengers[no_wifi_cols])[:, 1]

answers = pd.DataFrame({
    "chance satisfied, with wifi (%)": chance_wifi * 100,
    "chance satisfied, without wifi (%)": chance_no_wifi * 100,
    "answer, with wifi": model_wifi.predict(passengers[wifi_cols]),
    "answer, without wifi": model_no_wifi.predict(passengers[no_wifi_cols]),
}, index=passengers.index)

answers.round(1)

,"chance satisfied, with wifi (%)","chance satisfied, without wifi (%)","answer, with wifi","answer, without wifi"
happy,97.8,95.7,1,1
unhappy,0.0,1.0,0,0
middle,3.8,3.7,0,0


#### the wifi rating

In [5]:
# the same "middle" passenger, changing only the wifi rating
middle = passengers.loc[["middle"]]
rows = []

for wifi_score in [0, 1, 2, 3, 4, 5]:
    trial = middle.copy()
    trial["inflight_wifi_service"] = wifi_score

    rows.append({
        "wifi rating": wifi_score,
        "with wifi model (%)": model_wifi.predict_proba(trial[wifi_cols])[0, 1] * 100,
        "without wifi model (%)": model_no_wifi.predict_proba(trial[no_wifi_cols])[0, 1] * 100,
    })

pd.DataFrame(rows).round(1)

,wifi rating,with wifi model (%),without wifi model (%)
0,0,99.9,3.7
1,1,3.9,3.7
2,2,5.3,3.7
3,3,3.8,3.7
4,4,25.5,3.7
5,5,99.0,3.7


#### values that models never saw

In [6]:
# values the models did not see in training
odd = passengers.loc[["happy"]].copy()
print("Normal happy passenger        :", round(model_no_wifi.predict_proba(odd[no_wifi_cols])[0, 1] * 100, 1), "%")

odd["baggage_handling"] = 0
print("baggage rating of 0 (unseen)  :", round(model_no_wifi.predict_proba(odd[no_wifi_cols])[0, 1] * 100, 1), "%")

odd = passengers.loc[["happy"]].copy()
odd["age"] = 120
print("age of 120 (outside the data) :", round(model_no_wifi.predict_proba(odd[no_wifi_cols])[0, 1] * 100, 1), "%")

odd = passengers.loc[["happy"]].copy()
odd["flight_distance"] = 20000
print("20000 miles (outside the data):", round(model_no_wifi.predict_proba(odd[no_wifi_cols])[0, 1] * 100, 1), "%")

Normal happy passenger        : 95.7 %
baggage rating of 0 (unseen)  : 91.7 %
age of 120 (outside the data) : 82.1 %
20000 miles (outside the data): 95.2 %


#### checking what happen if having the broken input

In [8]:
happy = passengers.loc[["happy"], no_wifi_cols]

# 1. a column is missing
try:
    model_no_wifi.predict_proba(happy.drop(columns=["age"]))
except Exception as error:
    print("Missing column  :", type(error).__name__, "->", str(error)[:90])

# 2. one column too many
too_many = happy.copy()
too_many["gender"] = "Male"
try:
    result = model_no_wifi.predict_proba(too_many)[0, 1]
except Exception as error:
    print("Extra column    :", type(error).__name__, "->", str(error)[:90])
else:
    same = result == model_no_wifi.predict_proba(happy)[0, 1]
    print("Extra column    : no error,", "the extra column was ignored" if same else "the answer changed")

# 3. a typo in a label
typo = happy.copy()
typo["travel_class"] = "Economy plus"
try:
    model_no_wifi.predict_proba(typo)
except Exception as error:
    print("Typo in a label :", type(error).__name__, "->", str(error)[:90])

# 4. the columns in a different order
shuffled = happy[list(reversed(no_wifi_cols))]
same = model_no_wifi.predict_proba(shuffled)[0, 1] == model_no_wifi.predict_proba(happy)[0, 1]
print("Different order :", "same answer" if same else "different answer")

Missing column  : ValueError -> columns are missing: {'age'}
Extra column    : no error, the extra column was ignored
Typo in a label : ValueError -> Found unknown categories ['Economy plus'] in column 0 during transform
Different order : same answer


#### checking how fast one is predecting

In [9]:
start = time.time()
for i in range(100):
    model_no_wifi.predict_proba(happy)
one_prediction = (time.time() - start) / 100

print("One prediction takes about", round(one_prediction * 1000), "milliseconds")

One prediction takes about 3 milliseconds


#### checking the loaded models

In [10]:
import sys
from pathlib import Path
from sklearn.model_selection import train_test_split

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.db import get_engine, fetch_table
from src.features import make_features

df = fetch_table(get_engine())
X = make_features(df)
y = (df["satisfaction"] == "Satisfied").astype(int)

# only the training rows are used here; the test rows stay untouched
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("X_train:", X_train.shape)

X_train: (103904, 18)


#### checking the round trip

In [12]:
train_acc = model_wifi.score(X_train[wifi_cols], y_train) * 100
train_acc_nw = model_no_wifi.score(X_train[no_wifi_cols], y_train) * 100

print("With wifi    : training accuracy", round(train_acc, 2), "| from best model 96.92 |", round(train_acc, 2) == 96.92)
print("Without wifi : training accuracy", round(train_acc_nw, 2), "| from best model 95.08 |", round(train_acc_nw, 2) == 95.08)

With wifi    : training accuracy 96.92 | from best model 96.92 | True
Without wifi : training accuracy 95.08 | from best model 95.08 | True
